In [38]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [39]:
import importlib
import src.similarity_engine

importlib.reload(src.similarity_engine)

<module 'src.similarity_engine' from '/Users/Pietromiragoli/github-projects:/champions-league-player-discovery/src/similarity_engine.py'>

In [40]:
from src.similarity_engine import (
    find_similar_non_ucl_players,
    find_best_replacements
)

from src.scouting_score import (
    add_scouting_score
)

In [41]:
results = find_similar_non_ucl_players(
    "Julián Álvarez",
    striker_data,
    model,
    X_scaled,
    n_neighbors=15
)

scouting_results = add_scouting_score(results)

scouting_results[
    [
        "Player", "Age", "Squad", "Comp", "MP", "Min",
        "similarity_score", "minutes_score",
        "age_score", "scouting_score"
    ]
].head(10)

,Player,Age,Squad,Comp,MP,Min,similarity_score,minutes_score,age_score,scouting_score
0,Ragnar Ache,27.0,Köln,de Bundesliga,29,1718,66.522074,69.218372,70,67.274311
1,Akor Adams,26.0,Sevilla,es La Liga,28,1846,61.140245,74.375504,85,65.511509
107,Rômulo,24.0,RB Leipzig,de Bundesliga,28,2080,57.722507,83.803384,85,64.362388
73,Rafael Leão,26.0,Milan,it Serie A,27,1771,53.466080,71.353747,85,59.302622
103,Andrea Pinamonti,26.0,Sassuolo,it Serie A,34,2434,47.321055,98.066076,85,58.700703
96,Mikel Oyarzabal,29.0,Real Sociedad,es La Liga,30,2482,48.017600,100.000000,70,58.013200
43,Karl Etta Eyong,22.0,Levante,es La Liga,28,1410,50.255468,56.809025,100,56.212954
40,Emersonn,21.0,Toulouse,fr Ligue 1,26,1469,48.804904,59.186140,100,55.481599
29,Keinan Davis,28.0,Udinese,it Serie A,27,1890,48.566644,76.148268,70,54.847223
117,Sambou Soumano,25.0,Lorient,fr Ligue 1,30,931,53.075071,37.510073,85,53.932814


In [42]:
def add_scouting_score(results):
    results = results.copy()
    
    # Age handling: if Age exists, extract numeric age
    if "Age" in results.columns:
        results["Age_clean"] = (
            results["Age"]
            .astype(str)
            .str.extract(r"(\d+)")
            .astype(float)
        )
        
        results["age_score"] = results["Age_clean"].apply(
            lambda age: 100 if age <= 23 else
                        85 if age <= 26 else
                        70 if age <= 29 else
                        50
        )
    else:
        results["age_score"] = 70
    
    # Minutes reliability score
    results["minutes_score"] = (
        results["Min"] / results["Min"].max() * 100
    )
    
    # Final scouting score
    results["scouting_score"] = (
        0.60 * results["similarity_score"] +
        0.25 * results["minutes_score"] +
        0.15 * results["age_score"]
    )
    
    return results.sort_values("scouting_score", ascending=False)

In [43]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("../data/raw/players_data_light-2025_2026.csv")

ucl_teams = [
    "Arsenal", "Athletic Club", "Atlético Madrid", "Atalanta",
    "Barcelona", "Bayern Munich", "Chelsea", "Dortmund",
    "Eintracht Frankfurt", "Inter", "Juventus", "Leverkusen",
    "Liverpool", "Manchester City", "Marseille", "Monaco",
    "Napoli", "Newcastle United", "Paris Saint-Germain",
    "Real Madrid", "Tottenham Hotspur", "Villarreal"
]

df["is_ucl_team"] = df["Squad"].isin(ucl_teams)

df = df[(df["MP"] >= 15) & (df["Min"] >= 900)].copy()

strikers = df[df["Pos"] == "FW"].copy()

striker_features = ["Gls", "Ast", "G+A", "Sh", "SoT", "G/Sh"]

striker_data = strikers[
    [
        "Player", "Nation", "Squad", "Comp", "Age",
        "MP", "Min", "90s", "is_ucl_team"
    ] + striker_features
].copy()

striker_data["Gls_per90"] = striker_data["Gls"] / striker_data["90s"]
striker_data["Ast_per90"] = striker_data["Ast"] / striker_data["90s"]
striker_data["Sh_per90"] = striker_data["Sh"] / striker_data["90s"]
striker_data["SoT_per90"] = striker_data["SoT"] / striker_data["90s"]

advanced_striker_features = [
    "Gls_per90",
    "Ast_per90",
    "Sh_per90",
    "SoT_per90",
    "G/Sh"
]

striker_data = striker_data.reset_index(drop=True)

X = striker_data[advanced_striker_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = NearestNeighbors(
    n_neighbors=15,
    metric="euclidean"
)

model.fit(X_scaled)

striker_data.head()

,Player,Nation,Squad,Comp,Age,MP,Min,90s,is_ucl_team,Gls,Ast,G+A,Sh,SoT,G/Sh,Gls_per90,Ast_per90,Sh_per90,SoT_per90
0,Ragnar Ache,de GER,Köln,de Bundesliga,27.0,29,1718,19.1,False,7,4,11,52,22,0.13,0.366492,0.209424,2.722513,1.151832
1,Akor Adams,ng NGA,Sevilla,es La Liga,26.0,28,1846,20.5,False,8,3,11,55,27,0.09,0.390244,0.146341,2.682927,1.317073
2,Che Adams,sct SCO,Torino,it Serie A,29.0,31,1826,20.3,False,5,2,7,48,14,0.10,0.246305,0.098522,2.364532,0.689655
3,Ludovic Ajorque,fr FRA,Brest,fr Ligue 1,32.0,29,2550,28.3,False,7,9,16,47,15,0.15,0.247350,0.318021,1.660777,0.530035
4,Alexandre Alemão,br BRA,Rayo Vallecano,es La Liga,28.0,22,969,10.8,False,2,0,2,19,9,0.11,0.185185,0.000000,1.759259,0.833333


In [44]:
def find_similar_non_ucl_players(player_name, n_neighbors=15):
    player_match = striker_data[
        striker_data["Player"] == player_name
    ]

    if player_match.empty:
        return f"Player '{player_name}' not found."

    player_index = player_match.index[0]

    distances, indices = model.kneighbors(
        [X_scaled[player_index]],
        n_neighbors=n_neighbors
    )

    similar_players = striker_data.iloc[indices[0]].copy()
    similar_players["distance"] = distances[0]
    similar_players["similarity_score"] = 100 / (1 + similar_players["distance"])

    hidden_targets = similar_players[
        similar_players["is_ucl_team"] == False
    ].copy()

    return hidden_targets.sort_values("similarity_score", ascending=False)

In [45]:
results = find_similar_non_ucl_players("Julián Álvarez", n_neighbors=15)
results

,Player,Nation,Squad,Comp,Age,MP,Min,90s,is_ucl_team,Gls,...,G+A,Sh,SoT,G/Sh,Gls_per90,Ast_per90,Sh_per90,SoT_per90,distance,similarity_score
0,Ragnar Ache,de GER,Köln,de Bundesliga,27.0,29,1718,19.1,False,7,...,11,52,22,0.13,0.366492,0.209424,2.722513,1.151832,0.503260,66.522074
1,Akor Adams,ng NGA,Sevilla,es La Liga,26.0,28,1846,20.5,False,8,...,11,55,27,0.09,0.390244,0.146341,2.682927,1.317073,0.635584,61.140245
107,Rômulo,br BRA,RB Leipzig,de Bundesliga,24.0,28,2080,23.1,False,9,...,13,61,27,0.15,0.389610,0.173160,2.640693,1.168831,0.732426,57.722507
73,Rafael Leão,pt POR,Milan,it Serie A,26.0,27,1771,19.7,False,9,...,12,60,23,0.12,0.456853,0.152284,3.045685,1.167513,0.870345,53.466080
117,Sambou Soumano,sn SEN,Lorient,fr Ligue 1,25.0,30,931,10.3,False,4,...,6,28,10,0.14,0.388350,0.194175,2.718447,0.970874,0.884124,53.075071
43,Karl Etta Eyong,cm CMR,Levante,es La Liga,22.0,28,1410,15.7,False,6,...,8,41,21,0.15,0.382166,0.127389,2.611465,1.337580,0.989833,50.255468
40,Emersonn,br BRA,Toulouse,fr Ligue 1,21.0,26,1469,16.3,False,6,...,8,51,19,0.12,0.368098,0.122699,3.128834,1.165644,1.048974,48.804904
29,Keinan Davis,eng ENG,Udinese,it Serie A,28.0,27,1890,21.0,False,10,...,13,45,22,0.13,0.476190,0.142857,2.142857,1.047619,1.059026,48.566644
96,Mikel Oyarzabal,es ESP,Real Sociedad,es La Liga,29.0,30,2482,27.6,False,14,...,17,74,34,0.11,0.507246,0.108696,2.681159,1.231884,1.082570,48.017600
103,Andrea Pinamonti,it ITA,Sassuolo,it Serie A,26.0,34,2434,27.0,False,8,...,11,70,27,0.11,0.296296,0.111111,2.592593,1.000000,1.113224,47.321055


In [46]:
def add_scouting_score(results):

    results = results.copy()

    # Age score
    results["age_score"] = results["Age"].apply(
        lambda x: 100 if x <= 23 else
                  85 if x <= 26 else
                  70 if x <= 29 else
                  50
    )

    # Minutes score
    results["minutes_score"] = (
        results["Min"] / results["Min"].max()
    ) * 100

    # Final scouting score
    results["scouting_score"] = (
        0.60 * results["similarity_score"]
        + 0.25 * results["minutes_score"]
        + 0.15 * results["age_score"]
    )

    return results.sort_values(
        "scouting_score",
        ascending=False
    )

In [47]:
results = find_similar_non_ucl_players(
    "Julián Álvarez",
    n_neighbors=15
)

scouting_results = add_scouting_score(results)

scouting_results[
    [
        "Player",
        "Age",
        "Squad",
        "Comp",
        "MP",
        "Min",
        "similarity_score",
        "scouting_score"
    ]
]

,Player,Age,Squad,Comp,MP,Min,similarity_score,scouting_score
107,Rômulo,24.0,RB Leipzig,de Bundesliga,28,2080,57.722507,68.334350
1,Akor Adams,26.0,Sevilla,es La Liga,28,1846,61.140245,68.028023
0,Ragnar Ache,27.0,Köln,de Bundesliga,29,1718,66.522074,67.717837
103,Andrea Pinamonti,26.0,Sassuolo,it Serie A,34,2434,47.321055,65.659152
96,Mikel Oyarzabal,29.0,Real Sociedad,es La Liga,30,2482,48.017600,64.310560
73,Rafael Leão,26.0,Milan,it Serie A,27,1771,53.466080,62.668085
43,Karl Etta Eyong,22.0,Levante,es La Liga,28,1410,50.255468,59.355537
40,Emersonn,21.0,Toulouse,fr Ligue 1,26,1469,48.804904,59.079477
29,Keinan Davis,28.0,Udinese,it Serie A,27,1890,48.566644,58.677053
104,Christian Pulisic,27.0,Milan,it Serie A,28,1549,46.508658,54.007531


In [48]:
scouting_results.to_csv(
    "../data/processed/julian_alvarez_scouting_score_results.csv",
    index=False
)

scouting_results.head(10)

,Player,Nation,Squad,Comp,Age,MP,Min,90s,is_ucl_team,Gls,...,G/Sh,Gls_per90,Ast_per90,Sh_per90,SoT_per90,distance,similarity_score,age_score,minutes_score,scouting_score
107,Rômulo,br BRA,RB Leipzig,de Bundesliga,24.0,28,2080,23.1,False,9,...,0.15,0.389610,0.173160,2.640693,1.168831,0.732426,57.722507,85,83.803384,68.334350
1,Akor Adams,ng NGA,Sevilla,es La Liga,26.0,28,1846,20.5,False,8,...,0.09,0.390244,0.146341,2.682927,1.317073,0.635584,61.140245,85,74.375504,68.028023
0,Ragnar Ache,de GER,Köln,de Bundesliga,27.0,29,1718,19.1,False,7,...,0.13,0.366492,0.209424,2.722513,1.151832,0.503260,66.522074,70,69.218372,67.717837
103,Andrea Pinamonti,it ITA,Sassuolo,it Serie A,26.0,34,2434,27.0,False,8,...,0.11,0.296296,0.111111,2.592593,1.000000,1.113224,47.321055,85,98.066076,65.659152
96,Mikel Oyarzabal,es ESP,Real Sociedad,es La Liga,29.0,30,2482,27.6,False,14,...,0.11,0.507246,0.108696,2.681159,1.231884,1.082570,48.017600,70,100.000000,64.310560
73,Rafael Leão,pt POR,Milan,it Serie A,26.0,27,1771,19.7,False,9,...,0.12,0.456853,0.152284,3.045685,1.167513,0.870345,53.466080,85,71.353747,62.668085
43,Karl Etta Eyong,cm CMR,Levante,es La Liga,22.0,28,1410,15.7,False,6,...,0.15,0.382166,0.127389,2.611465,1.337580,0.989833,50.255468,100,56.809025,59.355537
40,Emersonn,br BRA,Toulouse,fr Ligue 1,21.0,26,1469,16.3,False,6,...,0.12,0.368098,0.122699,3.128834,1.165644,1.048974,48.804904,100,59.186140,59.079477
29,Keinan Davis,eng ENG,Udinese,it Serie A,28.0,27,1890,21.0,False,10,...,0.13,0.476190,0.142857,2.142857,1.047619,1.059026,48.566644,70,76.148268,58.677053
104,Christian Pulisic,us USA,Milan,it Serie A,27.0,28,1549,17.2,False,8,...,0.16,0.465116,0.174419,2.906977,1.395349,1.150137,46.508658,70,62.409347,54.007531


In [49]:
best_replacements = find_best_replacements(
    "Julián Álvarez",
    striker_data,
    model,
    X_scaled,
    top_n=5
)

best_replacements[
    [
        "Player",
        "Age",
        "Squad",
        "Comp",
        "similarity_score",
        "scouting_score"
    ]
]

,Player,Age,Squad,Comp,similarity_score,scouting_score
0,Ragnar Ache,27.0,Köln,de Bundesliga,66.522074,67.274311
1,Akor Adams,26.0,Sevilla,es La Liga,61.140245,65.511509
107,Rômulo,24.0,RB Leipzig,de Bundesliga,57.722507,64.362388
73,Rafael Leão,26.0,Milan,it Serie A,53.466080,59.302622
103,Andrea Pinamonti,26.0,Sassuolo,it Serie A,47.321055,58.700703


In [50]:
best_replacements[
    [
        "Player",
        "Age",
        "Min",
        "similarity_score",
        "minutes_score",
        "age_score",
        "scouting_score"
    ]
]

,Player,Age,Min,similarity_score,minutes_score,age_score,scouting_score
0,Ragnar Ache,27.0,1718,66.522074,69.218372,70,67.274311
1,Akor Adams,26.0,1846,61.140245,74.375504,85,65.511509
107,Rômulo,24.0,2080,57.722507,83.803384,85,64.362388
73,Rafael Leão,26.0,1771,53.466080,71.353747,85,59.302622
103,Andrea Pinamonti,26.0,2434,47.321055,98.066076,85,58.700703


In [51]:
import importlib

import src.scouting_score

importlib.reload(src.scouting_score)

<module 'src.scouting_score' from '/Users/Pietromiragoli/github-projects:/champions-league-player-discovery/src/scouting_score.py'>

In [1]:
def find_hidden_gems(
    striker_data,
    max_age=24,
    min_minutes=1200,
    top_n=10
):

    gems = striker_data.copy()

    gems = gems[
        (gems["is_ucl_team"] == False)
        & (gems["Age"] <= max_age)
        & (gems["Min"] >= min_minutes)
    ]

    gems = gems.sort_values(
        "Gls_per90",
        ascending=False
    )

    return gems.head(top_n)